# Data Quality Report: Raw, daily_rv, and event_5_min

This notebook audits:
1. Raw 5-minute data quality per coin and in total.
2. Interim `daily_rv` parquet quality per coin.
3. Interim `event_5_min` parquet quality per coin.

It reports:
- row counts per coin and global totals,
- missing-value diagnostics,
- date coverage and gap/incomplete-day counts,
- event and exceedance daily means (downside only).

In [17]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

from src.config.paths import (
    RAW_DATA_DIR,
    INTERIM_DAILY_RV_DIR,
    INTERIM_EVENT_5_MIN_DIR,
 )

EXPECTED_5MIN_PER_DAY = 288


def fmt_date(ts: pd.Timestamp | None) -> str:
    if ts is None or pd.isna(ts):
        return "NA"
    return pd.Timestamp(ts).strftime("%Y-%m-%d")


def expected_daily_index(start: pd.Timestamp, end: pd.Timestamp) -> pd.DatetimeIndex:
    return pd.date_range(start=start.floor("D"), end=end.floor("D"), freq="D")


def parse_coin_from_raw_filename(path: Path) -> str:
    match = re.match(r"([A-Z]+)USDT_.*\\.csv$", path.name)
    return match.group(1) if match else path.stem


def parse_coin_from_parquet_filename(path: Path, suffix: str) -> str:
    return path.name.replace(suffix, "")


print(f"RAW_DATA_DIR={RAW_DATA_DIR}")
print(f"INTERIM_DAILY_RV_DIR={INTERIM_DAILY_RV_DIR}")
print(f"INTERIM_EVENT_5_MIN_DIR={INTERIM_EVENT_5_MIN_DIR}")

RAW_DATA_DIR=C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\Hawkes Enhanced RV Forecast\data\raw
INTERIM_DAILY_RV_DIR=C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\Hawkes Enhanced RV Forecast\data\interim\daily_rv
INTERIM_EVENT_5_MIN_DIR=C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\Hawkes Enhanced RV Forecast\data\interim\event_5_min


## Section 1: Raw 5-Minute Data Quality

In [18]:
raw_files = sorted(RAW_DATA_DIR.glob("*USDT*_2026.csv"))

if not raw_files:
    raise FileNotFoundError(f"No raw CSV files found in {RAW_DATA_DIR}")

raw_summary_rows = []

for raw_path in raw_files:
    coin = parse_coin_from_raw_filename(raw_path)

    df = pd.read_csv(raw_path)
    row_count = len(df)
    missing_cells = int(df.isna().sum().sum())
    rows_with_any_na = int(df.isna().any(axis=1).sum())

    if "open_time" not in df.columns:
        raise ValueError(f"{raw_path.name} is missing required column: open_time")

    ts = pd.to_datetime(df["open_time"], errors="coerce")
    invalid_open_time = int(ts.isna().sum())
    ts_valid = ts.dropna()

    if ts_valid.empty:
        date_start = None
        date_end = None
        expected_days = 0
        missing_days = 0
        incomplete_days = 0
        pct_missing_days = np.nan
        pct_incomplete_days = np.nan
    else:
        day_counts = ts_valid.dt.floor("D").value_counts().sort_index()
        date_start = day_counts.index.min()
        date_end = day_counts.index.max()

        expected_days_idx = expected_daily_index(date_start, date_end)
        expected_days = len(expected_days_idx)

        missing_days = int(expected_days - len(day_counts.index))
        incomplete_days = int((day_counts < EXPECTED_5MIN_PER_DAY).sum())

        pct_missing_days = 100.0 * missing_days / expected_days if expected_days else np.nan
        pct_incomplete_days = 100.0 * incomplete_days / expected_days if expected_days else np.nan

    raw_summary_rows.append(
        {
            "coin": coin,
            "file": raw_path.name,
            "rows": row_count,
            "missing_cells": missing_cells,
            "rows_with_any_na": rows_with_any_na,
            "invalid_open_time": invalid_open_time,
            "date_start": fmt_date(date_start),
            "date_end": fmt_date(date_end),
            "expected_days": expected_days,
            "missing_days": missing_days,
            "incomplete_days": incomplete_days,
            "pct_missing_days": pct_missing_days,
            "pct_incomplete_days": pct_incomplete_days,
        }
    )

raw_summary = pd.DataFrame(raw_summary_rows).sort_values("coin").reset_index(drop=True)

print("RAW: per-coin summary")
display(raw_summary)

print("\nRAW: quality sentence per coin")
for _, r in raw_summary.iterrows():
    print(
        f"- {r['coin']}: {r['date_start']} to {r['date_end']} "
        f"with {int(r['missing_days'])} missing days and {int(r['incomplete_days'])} days with incomplete data"
    )

raw_totals = {
    "coins": int(raw_summary['coin'].nunique()),
    "total_rows": int(raw_summary['rows'].sum()),
    "total_missing_cells": int(raw_summary['missing_cells'].sum()),
    "total_rows_with_any_na": int(raw_summary['rows_with_any_na'].sum()),
    "total_invalid_open_time": int(raw_summary['invalid_open_time'].sum()),
    "total_expected_days": int(raw_summary['expected_days'].sum()),
    "total_missing_days": int(raw_summary['missing_days'].sum()),
    "total_incomplete_days": int(raw_summary['incomplete_days'].sum()),
}

raw_totals['pct_missing_days_global'] = (
    100.0 * raw_totals['total_missing_days'] / raw_totals['total_expected_days']
    if raw_totals['total_expected_days'] else np.nan
)
raw_totals['pct_incomplete_days_global'] = (
    100.0 * raw_totals['total_incomplete_days'] / raw_totals['total_expected_days']
    if raw_totals['total_expected_days'] else np.nan
)

print("\nRAW: global totals")
for k, v in raw_totals.items():
    if isinstance(v, float):
        print(f"- {k}: {v:.4f}")
    else:
        print(f"- {k}: {v}")

RAW: per-coin summary


,coin,file,rows,missing_cells,rows_with_any_na,invalid_open_time,date_start,date_end,expected_days,missing_days,incomplete_days,pct_missing_days,pct_incomplete_days
0,ADAUSDT_binance_5min_2017_2026,ADAUSDT_binance_5min_2017_2026.csv,809785,0,0,0,2018-04-17,2026-01-01,2817,0,28,0.0,0.993965
1,BTCUSDT_binance_5min_2017_2026,BTCUSDT_binance_5min_2017_2026.csv,879230,0,0,0,2017-08-17,2026-01-01,3060,0,36,0.0,1.176471
2,ETHUSDT_binance_5min_2017_2026,ETHUSDT_binance_5min_2017_2026.csv,879230,0,0,0,2017-08-17,2026-01-01,3060,0,36,0.0,1.176471
3,SOLUSDT_binance_5min_2017_2026,SOLUSDT_binance_5min_2017_2026.csv,566714,0,0,0,2020-08-11,2026-01-01,1970,0,12,0.0,0.609137
4,XRPUSDT_binance_5min_2017_2026,XRPUSDT_binance_5min_2017_2026.csv,804839,0,0,0,2018-05-04,2026-01-01,2800,0,28,0.0,1.000000



RAW: quality sentence per coin
- ADAUSDT_binance_5min_2017_2026: 2018-04-17 to 2026-01-01 with 0 missing days and 28 days with incomplete data
- BTCUSDT_binance_5min_2017_2026: 2017-08-17 to 2026-01-01 with 0 missing days and 36 days with incomplete data
- ETHUSDT_binance_5min_2017_2026: 2017-08-17 to 2026-01-01 with 0 missing days and 36 days with incomplete data
- SOLUSDT_binance_5min_2017_2026: 2020-08-11 to 2026-01-01 with 0 missing days and 12 days with incomplete data
- XRPUSDT_binance_5min_2017_2026: 2018-05-04 to 2026-01-01 with 0 missing days and 28 days with incomplete data

RAW: global totals
- coins: 5
- total_rows: 3939798
- total_missing_cells: 0
- total_rows_with_any_na: 0
- total_invalid_open_time: 0
- total_expected_days: 13707
- total_missing_days: 0
- total_incomplete_days: 140
- pct_missing_days_global: 0.0000
- pct_incomplete_days_global: 1.0214


## Section 2: Interim daily_rv Parquet Quality

In [19]:
daily_files = sorted(INTERIM_DAILY_RV_DIR.glob("*_rv.parquet"))

if not daily_files:
    raise FileNotFoundError(f"No daily_rv parquet files found in {INTERIM_DAILY_RV_DIR}")

daily_summary_rows = []

for p in daily_files:
    coin = parse_coin_from_parquet_filename(p, "_rv.parquet")
    df = pd.read_parquet(p)

    rows = len(df)
    missing_cells = int(df.isna().sum().sum())
    rows_with_any_na = int(df.isna().any(axis=1).sum()) if rows > 0 else 0

    if rows == 0:
        date_start = None
        date_end = None
        expected_days = 0
        missing_days = 0
        incomplete_days = 0
    else:
        if isinstance(df.index, pd.DatetimeIndex):
            dt_idx = pd.to_datetime(df.index, errors="coerce")
        elif "date" in df.columns:
            dt_idx = pd.to_datetime(df["date"], errors="coerce")
        elif "time" in df.columns:
            dt_idx = pd.to_datetime(df["time"], errors="coerce")
        else:
            dt_idx = pd.to_datetime(df.index, errors="coerce")

        dt_idx = pd.Series(dt_idx).dropna()

        if dt_idx.empty:
            date_start = None
            date_end = None
            expected_days = 0
            missing_days = 0
            incomplete_days = 0
        else:
            day_counts = dt_idx.dt.floor("D").value_counts().sort_index()
            date_start = day_counts.index.min()
            date_end = day_counts.index.max()

            expected_days_idx = expected_daily_index(date_start, date_end)
            expected_days = len(expected_days_idx)
            missing_days = int(expected_days - len(day_counts.index))
            # For daily_rv, we expect 1 row/day. Any other count is flagged as incomplete.
            incomplete_days = int((day_counts != 1).sum())

    daily_summary_rows.append(
        {
            "coin": coin,
            "file": p.name,
            "rows": rows,
            "missing_cells": missing_cells,
            "rows_with_any_na": rows_with_any_na,
            "date_start": fmt_date(date_start),
            "date_end": fmt_date(date_end),
            "expected_days": expected_days,
            "missing_days": missing_days,
            "incomplete_days": incomplete_days,
        }
    )

daily_summary = pd.DataFrame(daily_summary_rows).sort_values("coin").reset_index(drop=True)

print("DAILY_RV: per-coin summary")
display(daily_summary)

print("\nDAILY_RV: parquet quality sentence per coin")
for _, r in daily_summary.iterrows():
    print(
        f"- {r['file']}: {r['date_start']} to {r['date_end']} "
        f"with {int(r['missing_days'])} missing days and {int(r['incomplete_days'])} days with incomplete data"
    )

print("\nDAILY_RV: global totals")
print(f"- total coins: {daily_summary['coin'].nunique()}")
print(f"- total rows: {int(daily_summary['rows'].sum())}")
print(f"- total missing cells: {int(daily_summary['missing_cells'].sum())}")
print(f"- total rows with any NA: {int(daily_summary['rows_with_any_na'].sum())}")
print(f"- total missing days: {int(daily_summary['missing_days'].sum())}")
print(f"- total incomplete days: {int(daily_summary['incomplete_days'].sum())}")

DAILY_RV: per-coin summary


,coin,file,rows,missing_cells,rows_with_any_na,date_start,date_end,expected_days,missing_days,incomplete_days
0,BTC,BTC_rv.parquet,3059,55,53,2017-08-17,2025-12-31,3059,0,0
1,ETH,ETH_rv.parquet,3059,55,53,2017-08-17,2025-12-31,3059,0,0



DAILY_RV: parquet quality sentence per coin
- BTC_rv.parquet: 2017-08-17 to 2025-12-31 with 0 missing days and 0 days with incomplete data
- ETH_rv.parquet: 2017-08-17 to 2025-12-31 with 0 missing days and 0 days with incomplete data

DAILY_RV: global totals
- total coins: 2
- total rows: 6118
- total missing cells: 110
- total rows with any NA: 106
- total missing days: 0
- total incomplete days: 0


## Section 3: Interim event_5_min Parquet Quality and Event Statistics

In [20]:
event_files = sorted(INTERIM_EVENT_5_MIN_DIR.glob("*_5m_events.parquet"))

if not event_files:
    raise FileNotFoundError(f"No event_5_min parquet files found in {INTERIM_EVENT_5_MIN_DIR}")

event_summary_rows = []
all_daily_events = []

for p in event_files:
    coin = parse_coin_from_parquet_filename(p, "_5m_events.parquet")
    df = pd.read_parquet(p)

    rows = len(df)
    missing_cells = int(df.isna().sum().sum())
    rows_with_any_na = int(df.isna().any(axis=1).sum()) if rows > 0 else 0

    if "time" not in df.columns:
        raise ValueError(f"{p.name} is missing required column: time")

    ts = pd.to_datetime(df["time"], errors="coerce")
    invalid_time = int(ts.isna().sum())
    ts_valid = ts.dropna()

    # Handle both new and legacy schemas.
    if "Event_down" in df.columns:
        event_down = pd.to_numeric(df["Event_down"], errors="coerce").fillna(0.0)
    elif "Event" in df.columns:
        event_down = pd.to_numeric(df["Event"], errors="coerce").fillna(0.0)
    else:
        event_down = pd.Series(np.zeros(rows), index=df.index, dtype="float64")

    if ts_valid.empty:
        date_start = None
        date_end = None
        expected_days = 0
        missing_days = 0
        incomplete_days = 0
        mean_daily_events_down = np.nan
        mean_daily_exceedance_down = np.nan
    else:
        date_series = ts.dt.floor("D")
        day_counts = date_series.value_counts().sort_index()

        date_start = day_counts.index.min()
        date_end = day_counts.index.max()

        expected_days_idx = expected_daily_index(date_start, date_end)
        expected_days = len(expected_days_idx)
        missing_days = int(expected_days - len(day_counts.index))
        incomplete_days = int((day_counts < EXPECTED_5MIN_PER_DAY).sum())

        by_day = pd.DataFrame({
            "date": date_series,
            "event_down": event_down,
        }).dropna(subset=["date"])

        daily_agg = by_day.groupby("date", as_index=False).agg(
            n_events_down=("event_down", lambda x: int((x > 0).sum())),
            exceedance_down=("event_down", "mean"),
        )
        daily_agg["coin"] = coin
        all_daily_events.append(daily_agg)

        mean_daily_events_down = float(daily_agg["n_events_down"].mean())
        mean_daily_exceedance_down = float(daily_agg["exceedance_down"].mean())

    event_summary_rows.append(
        {
            "coin": coin,
            "file": p.name,
            "rows": rows,
            "missing_cells": missing_cells,
            "rows_with_any_na": rows_with_any_na,
            "invalid_time": invalid_time,
            "date_start": fmt_date(date_start),
            "date_end": fmt_date(date_end),
            "expected_days": expected_days,
            "missing_days": missing_days,
            "incomplete_days": incomplete_days,
            "mean_daily_events_down": mean_daily_events_down,
            "mean_daily_exceedance_down": mean_daily_exceedance_down,
        }
    )

event_summary = pd.DataFrame(event_summary_rows).sort_values("coin").reset_index(drop=True)

N_EVENT_COLS = ["mean_daily_events_down"]
EXCEEDANCE_COLS = ["mean_daily_exceedance_down"]
EXCEEDANCE_SCALE = 1e-4

event_summary_display = event_summary.copy()
for col in N_EVENT_COLS:
    event_summary_display[col] = event_summary_display[col].astype(float).round(2)
for col in EXCEEDANCE_COLS:
    event_summary_display[col] = (event_summary_display[col].astype(float) / EXCEEDANCE_SCALE).round(2)

event_summary_display = event_summary_display.rename(
    columns={col: f"{col}_x1e4" for col in EXCEEDANCE_COLS}
)

print("EVENT_5_MIN: per-coin quality + event/exceedance means")
print("(n_events shown as xx.xx, exceedance shown in x1e-4 units)")
display(event_summary_display)

print("\nEVENT_5_MIN: parquet quality sentence per coin")
for _, r in event_summary.iterrows():
    print(
        f"- {r['file']}: {r['date_start']} to {r['date_end']} "
        f"with {int(r['missing_days'])} missing days and {int(r['incomplete_days'])} days with incomplete data"
    )

if all_daily_events:
    daily_events_all = pd.concat(all_daily_events, ignore_index=True)

    print("\nEVENT_5_MIN: mean daily number of events (per coin)")
    per_coin_events = daily_events_all.groupby("coin", as_index=False).agg(
        mean_n_events_down=("n_events_down", "mean"),
        mean_exceedance_down=("exceedance_down", "mean"),
    ).sort_values("coin")

    per_coin_events_display = per_coin_events.copy()
    for col in ["mean_n_events_down"]:
        per_coin_events_display[col] = per_coin_events_display[col].astype(float).round(2)
    for col in ["mean_exceedance_down"]:
        per_coin_events_display[col] = (per_coin_events_display[col].astype(float) / EXCEEDANCE_SCALE).round(2)

    per_coin_events_display = per_coin_events_display.rename(
        columns={"mean_exceedance_down": "mean_exceedance_down_x1e4"}
    )

    print("(n_events shown as xx.xx, exceedance shown in x1e-4 units)")
    display(per_coin_events_display)

    print("\nEVENT_5_MIN: global means across all coin-days")
    global_means = {
        "mean_n_events_down": float(daily_events_all["n_events_down"].mean()),
        "mean_exceedance_down": float(daily_events_all["exceedance_down"].mean()),
    }

    print(f"- mean_n_events_down: {global_means['mean_n_events_down']:.2f}")
    print(f"- mean_exceedance_down_x1e4: {global_means['mean_exceedance_down'] / EXCEEDANCE_SCALE:.2f}")

print("\nEVENT_5_MIN: global totals")
print(f"- total coins: {event_summary['coin'].nunique()}")
print(f"- total rows: {int(event_summary['rows'].sum())}")
print(f"- total missing cells: {int(event_summary['missing_cells'].sum())}")
print(f"- total rows with any NA: {int(event_summary['rows_with_any_na'].sum())}")
print(f"- total invalid_time: {int(event_summary['invalid_time'].sum())}")
print(f"- total missing days: {int(event_summary['missing_days'].sum())}")
print(f"- total incomplete days: {int(event_summary['incomplete_days'].sum())}")

EVENT_5_MIN: per-coin quality + event/exceedance means
(n_events shown as xx.xx, exceedance shown in x1e-4 units)


,coin,file,rows,missing_cells,rows_with_any_na,invalid_time,date_start,date_end,expected_days,missing_days,incomplete_days,mean_daily_events_down,mean_daily_exceedance_down_x1e4
0,BTC,BTC_5m_events.parquet,877262,8557,8557,0,2017-08-24,2026-01-01,3053,0,35,0.00,0.00
1,ETH,ETH_5m_events.parquet,877262,8557,8557,0,2017-08-24,2026-01-01,3053,0,35,15.17,1.01



EVENT_5_MIN: parquet quality sentence per coin
- BTC_5m_events.parquet: 2017-08-24 to 2026-01-01 with 0 missing days and 35 days with incomplete data
- ETH_5m_events.parquet: 2017-08-24 to 2026-01-01 with 0 missing days and 35 days with incomplete data

EVENT_5_MIN: mean daily number of events (per coin)
(n_events shown as xx.xx, exceedance shown in x1e-4 units)


,coin,mean_n_events_down,mean_exceedance_down_x1e4
0,BTC,0.00,0.00
1,ETH,15.17,1.01



EVENT_5_MIN: global means across all coin-days
- mean_n_events_down: 7.58
- mean_exceedance_down_x1e4: 0.50

EVENT_5_MIN: global totals
- total coins: 2
- total rows: 1754524
- total missing cells: 17114
- total rows with any NA: 17114
- total invalid_time: 0
- total missing days: 0
- total incomplete days: 70


## Section 4: lambda_hawkes Alignment Between event_5_min and daily_rv

In [21]:
event_files = sorted(INTERIM_EVENT_5_MIN_DIR.glob("*_5m_events.parquet"))
daily_files = sorted(INTERIM_DAILY_RV_DIR.glob("*_rv.parquet"))

if not event_files:
    raise FileNotFoundError(f"No event_5_min parquet files found in {INTERIM_EVENT_5_MIN_DIR}")
if not daily_files:
    raise FileNotFoundError(f"No daily_rv parquet files found in {INTERIM_DAILY_RV_DIR}")

event_by_coin = {parse_coin_from_parquet_filename(p, "_5m_events.parquet"): p for p in event_files}
daily_by_coin = {parse_coin_from_parquet_filename(p, "_rv.parquet"): p for p in daily_files}
coins = sorted(set(event_by_coin).intersection(daily_by_coin))

if not coins:
    raise ValueError("No overlapping coins between event_5_min and daily_rv files.")

alignment_rows = []

for coin in coins:
    event_path = event_by_coin[coin]
    daily_path = daily_by_coin[coin]

    event_df = pd.read_parquet(event_path)
    daily_df = pd.read_parquet(daily_path)

    if "time" not in event_df.columns:
        raise ValueError(f"{event_path.name} is missing required column: time")

    event_has_lambda = "lambda_hawkes" in event_df.columns
    daily_has_lambda = "lambda_hawkes" in daily_df.columns

    event_time = pd.to_datetime(event_df["time"], errors="coerce")
    event_day_counts = event_time.dropna().dt.floor("D").value_counts().sort_index()

    if event_day_counts.empty:
        date_start = None
        date_end = None
        expected_days = 0
        missing_days = 0
        incomplete_days = 0
    else:
        date_start = event_day_counts.index.min()
        date_end = event_day_counts.index.max()
        expected_days_idx = expected_daily_index(date_start, date_end)
        expected_days = len(expected_days_idx)
        missing_days = int(expected_days - len(event_day_counts.index))
        incomplete_days = int((event_day_counts < EXPECTED_5MIN_PER_DAY).sum())

    if (not event_has_lambda) or (not daily_has_lambda):
        status_msg = []
        if not event_has_lambda:
            status_msg.append("missing lambda_hawkes in event_5_min")
        if not daily_has_lambda:
            status_msg.append("missing lambda_hawkes in daily_rv")

        alignment_rows.append(
            {
                "coin": coin,
                "event_file": event_path.name,
                "daily_file": daily_path.name,
                "date_start": fmt_date(date_start),
                "date_end": fmt_date(date_end),
                "expected_days": expected_days,
                "event_missing_days": missing_days,
                "event_incomplete_days": incomplete_days,
                "event_lambda_na": np.nan,
                "event_lambda_negative": np.nan,
                "daily_lambda_na": np.nan,
                "daily_lambda_negative": np.nan,
                "matched_days": 0,
                "mismatch_days": np.nan,
                "max_abs_diff": np.nan,
                "mean_abs_diff": np.nan,
                "max_within_day_spread_event": np.nan,
                "status": "; ".join(status_msg),
            }
        )
        continue

    event_lambda = pd.to_numeric(event_df["lambda_hawkes"], errors="coerce")
    event_tmp = pd.DataFrame({
        "date": event_time.dt.floor("D"),
        "lambda_hawkes": event_lambda,
    }).dropna(subset=["date"])

    by_day = event_tmp.groupby("date")["lambda_hawkes"]
    event_daily = by_day.mean()
    event_daily.name = "event_lambda_hawkes"

    spreads = by_day.max() - by_day.min()
    event_within_day_spread = float(spreads.max()) if len(spreads) else np.nan

    if isinstance(daily_df.index, pd.DatetimeIndex):
        daily_dates = pd.to_datetime(daily_df.index, errors="coerce")
    elif "date" in daily_df.columns:
        daily_dates = pd.to_datetime(daily_df["date"], errors="coerce")
    elif "time" in daily_df.columns:
        daily_dates = pd.to_datetime(daily_df["time"], errors="coerce")
    else:
        daily_dates = pd.to_datetime(daily_df.index, errors="coerce")

    daily_lambda = pd.to_numeric(daily_df["lambda_hawkes"], errors="coerce")
    daily_tmp = pd.DataFrame({
        "date": pd.Series(daily_dates).dt.floor("D"),
        "daily_lambda_hawkes": daily_lambda,
    }).dropna(subset=["date"])

    daily_daily = (
        daily_tmp
        .drop_duplicates(subset=["date"], keep="last")
        .set_index("date")["daily_lambda_hawkes"]
    )

    aligned = pd.concat([event_daily, daily_daily], axis=1, join="inner")
    if aligned.empty:
        matched_days = 0
        mismatch_days = 0
        max_abs_diff = np.nan
        mean_abs_diff = np.nan
    else:
        abs_diff = (aligned["event_lambda_hawkes"] - aligned["daily_lambda_hawkes"]).abs()
        matched_days = int(len(aligned))
        mismatch_days = int((abs_diff > 1e-12).sum())
        max_abs_diff = float(abs_diff.max())
        mean_abs_diff = float(abs_diff.mean())

    event_lambda_na = int(event_lambda.isna().sum())
    event_lambda_negative = int((event_lambda.dropna() < 0).sum())
    daily_lambda_na = int(daily_lambda.isna().sum())
    daily_lambda_negative = int((daily_lambda.dropna() < 0).sum())

    alignment_rows.append(
        {
            "coin": coin,
            "event_file": event_path.name,
            "daily_file": daily_path.name,
            "date_start": fmt_date(date_start),
            "date_end": fmt_date(date_end),
            "expected_days": expected_days,
            "event_missing_days": missing_days,
            "event_incomplete_days": incomplete_days,
            "event_lambda_na": event_lambda_na,
            "event_lambda_negative": event_lambda_negative,
            "daily_lambda_na": daily_lambda_na,
            "daily_lambda_negative": daily_lambda_negative,
            "matched_days": matched_days,
            "mismatch_days": mismatch_days,
            "max_abs_diff": max_abs_diff,
            "mean_abs_diff": mean_abs_diff,
            "max_within_day_spread_event": event_within_day_spread,
            "status": "ok",
        }
    )

alignment_summary = pd.DataFrame(alignment_rows).sort_values("coin").reset_index(drop=True)

alignment_display = alignment_summary.copy()
for col in ["max_abs_diff", "mean_abs_diff", "max_within_day_spread_event"]:
    alignment_display[col] = alignment_display[col].astype(float).round(12)

print("LAMBDA_HAWKES ALIGNMENT: event_5_min vs daily_rv")
display(alignment_display)

print("\nAlignment sentence per coin")
for _, r in alignment_summary.iterrows():
    print(
        f"- {r['coin']}: {r['date_start']} to {r['date_end']} | "
        f"status={r['status']} | matched_days={int(r['matched_days'])} | "
        f"event_missing_days={int(r['event_missing_days'])}, event_incomplete_days={int(r['event_incomplete_days'])}"
    )

print("\nAlignment global totals")
print(f"- total_coins: {alignment_summary['coin'].nunique()}")
print(f"- total_matched_days: {int(alignment_summary['matched_days'].sum())}")
print(f"- coins_with_missing_lambda_columns: {int((alignment_summary['status'] != 'ok').sum())}")

LAMBDA_HAWKES ALIGNMENT: event_5_min vs daily_rv


,coin,event_file,daily_file,date_start,date_end,expected_days,event_missing_days,event_incomplete_days,event_lambda_na,event_lambda_negative,daily_lambda_na,daily_lambda_negative,matched_days,mismatch_days,max_abs_diff,mean_abs_diff,max_within_day_spread_event,status
0,BTC,BTC_5m_events.parquet,BTC_rv.parquet,2017-08-24,2026-01-01,3053,0,35,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,missing lambda_hawkes in event_5_min; missing ...
1,ETH,ETH_5m_events.parquet,ETH_rv.parquet,2017-08-24,2026-01-01,3053,0,35,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,missing lambda_hawkes in event_5_min; missing ...



Alignment sentence per coin
- BTC: 2017-08-24 to 2026-01-01 | status=missing lambda_hawkes in event_5_min; missing lambda_hawkes in daily_rv | matched_days=0 | event_missing_days=0, event_incomplete_days=35
- ETH: 2017-08-24 to 2026-01-01 | status=missing lambda_hawkes in event_5_min; missing lambda_hawkes in daily_rv | matched_days=0 | event_missing_days=0, event_incomplete_days=35

Alignment global totals
- total_coins: 2
- total_matched_days: 0
- coins_with_missing_lambda_columns: 2
